# AŞAMA 3A — Test tahminlerini kilitleme için hazırlık

Seed 31040. Bu notebook yalnızca development verisiyle model eğitir ve **test etiketlerini okumadan** test tahminlerini üretir. Test başarı metrikleri 3B aşamasında, GitHub'da kilitleme commit'i doğrulandıktan sonra hesaplanacaktır.

Klasör: MyDrive/Yapay Sinir Ağları/YSA_31040_20261118. Notebook'un başarı mesajı GitHub kilidinin oluştuğu anlamına gelmez. Çıktı olarak kilit CSV, manifest ve nihai hiperparametreler JSON dosyasını ChatGPT'ye gönderin.


## 0 — Google Drive ve kitaplıklar


In [ ]:
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG",":4096:8")
import sys, copy, json, random, hashlib, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
import sklearn,joblib
if Path('/content').exists():
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT=Path('/content/drive/MyDrive/Yapay Sinir Ağları/YSA_31040_20261118')
else:
    ROOT=Path(os.environ.get('YSA_ROOT_OVERRIDE',str(Path.cwd()/'YSA_31040_20261118')))
DATA,RESULTS,CONFIGS,FIGURES=[ROOT/x for x in ('data','results','configs','figures')]
MODELS=ROOT/'models'
HISTORIES=RESULTS/'stage3_histories'
for p in [MODELS,HISTORIES]:p.mkdir(parents=True,exist_ok=True)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CODE_VERSION='stage3A-seed31040-v2'
def sha(path):return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def save_json(path,obj):
    tmp=Path(str(path)+'.part')
    tmp.write_text(json.dumps(obj,ensure_ascii=False,indent=2,allow_nan=False),encoding='utf-8')
    tmp.replace(path)
def write_csv(df,path):
    tmp=Path(str(path)+'.part')
    df.to_csv(tmp,index=False,float_format='%.17g',lineterminator='\n')
    tmp.replace(path)
def seed_everywhere(seed):
    random.seed(int(seed));np.random.seed(int(seed));torch.manual_seed(int(seed))
    if torch.cuda.is_available():torch.cuda.manual_seed_all(int(seed))
    torch.backends.cudnn.deterministic=True
    torch.backends.cudnn.benchmark=False
    torch.use_deterministic_algorithms(True,warn_only=True)
print('Aşama 3A | klasör:',ROOT,'| cihaz:',DEVICE)


## 1 — Aşama 2 dosyalarının ve veri bölümünün bütünlüğünü doğrula


In [ ]:
needed=[DATA/'bank-additional.csv',RESULTS/'dataset_split.csv',CONFIGS/'seeds.json',
    RESULTS/'raw_data_sha256.json',RESULTS/'stage1_sha256_manifest.json',
    RESULTS/'stage2_sha256_manifest.json',CONFIGS/'stage2_plan.json',
    CONFIGS/'final_hyperparameters.json',CONFIGS/'manual_candidates.json',
    CONFIGS/'random_candidates.json',RESULTS/'final_inner_search.csv',
    RESULTS/'outer_cv_results.csv',RESULTS/'removed_exact_duplicates.csv']
missing=[str(p) for p in needed if not p.is_file()]
if missing:raise FileNotFoundError('Aşama 2 dosyaları eksik: '+repr(missing))
SEEDS=json.loads((CONFIGS/'seeds.json').read_text(encoding='utf-8'))
FINAL=json.loads((CONFIGS/'final_hyperparameters.json').read_text(encoding='utf-8'))
PLAN=json.loads((CONFIGS/'stage2_plan.json').read_text(encoding='utf-8'))
MANIFEST2=json.loads((RESULTS/'stage2_sha256_manifest.json').read_text(encoding='utf-8'))
raw_sha=json.loads((RESULTS/'raw_data_sha256.json').read_text(encoding='utf-8'))['sha256']
assert SEEDS['base']==31040 and SEEDS['torch']==31077
assert FINAL['seed_base']==31040 and FINAL['stage2_plan_sha256']==PLAN['plan_sha256']
assert raw_sha==sha(DATA/'bank-additional.csv')==FINAL['raw_data_sha256']
assert sha(RESULTS/'dataset_split.csv')==FINAL['split_sha256']
assert MANIFEST2['stage2_plan_sha256']==PLAN['plan_sha256']
for rel,expected in MANIFEST2['files'].items():
    if rel.startswith('results/stage3_') or rel.startswith('configs/test_lock_'):continue
    p=ROOT/rel
    if not p.is_file() or sha(p)!=expected:raise RuntimeError('Aşama 2 hash kontrolü başarısız: '+rel)
assert FINAL['threshold']==0.5 and FINAL['selection_metric']=='average_precision'
SEED_LIST=[int(x) for x in FINAL['initialization_seeds']]
assert len(SEED_LIST)==3 and FINAL['primary_initialization_seed']==SEED_LIST[0]
CFG=FINAL['config']
assert CFG['id']==FINAL['candidate']
raw=pd.read_csv(DATA/'bank-additional.csv',sep=';')
assert raw.shape==(4119,21)
split=pd.read_csv(RESULTS/'dataset_split.csv')
removed=pd.read_csv(RESULTS/'removed_exact_duplicates.csv')
removed_ids=set(removed['removed_original_index'].astype(int)) if len(removed) else set()
assert split['original_index'].is_unique
assert set(split['original_index'])==set(raw.index)-removed_ids
DEV_IDS=split.loc[split['split']=='development','original_index'].astype(int).to_numpy()
TEST_IDS=split.loc[split['split']=='test','original_index'].astype(int).to_numpy()
assert len(DEV_IDS)>0 and len(TEST_IDS)>0 and not set(DEV_IDS)&set(TEST_IDS)
# Yalnızca development etiketlerini oku. TEST y sütununa kesinlikle erişilmez.
Y_DEV=raw.loc[DEV_IDS,'y'].map({'no':0,'yes':1}).astype(int).to_numpy()
raw_features=raw.drop(columns=['y','duration'])
X_DEV=raw_features.loc[DEV_IDS].copy()
X_TEST=raw_features.loc[TEST_IDS].copy()
NUM=X_DEV.select_dtypes(include='number').columns.tolist()
CAT=X_DEV.select_dtypes(exclude='number').columns.tolist()
assert all(c not in X_TEST.columns for c in ['y','duration'])
print('Aşama 2 dosyaları doğrulandı | aday:',FINAL['candidate'])
print('Development:',len(DEV_IDS),'test girdileri:',len(TEST_IDS),'seedler:',SEED_LIST)


## 2 — Aşama 2 ile aynı PyTorch MLP ve sızıntısız ön işleme


In [ ]:
class PercentileClipper(BaseEstimator,TransformerMixin):
    def __init__(self,column_names=(),enabled=False,q_low=0.01,q_high=0.99):
        self.column_names=column_names
        self.enabled=enabled
        self.q_low=q_low
        self.q_high=q_high
    def fit(self,Z,y=None):
        a=np.asarray(Z,dtype=float)
        self.lo_=np.full(a.shape[1],-np.inf)
        self.hi_=np.full(a.shape[1],np.inf)
        if self.enabled:
            for j,name in enumerate(self.column_names):
                if name=='pdays':continue
                self.lo_[j]=np.quantile(a[:,j],self.q_low)
                self.hi_[j]=np.quantile(a[:,j],self.q_high)
        return self
    def transform(self,Z):
        return np.clip(np.asarray(Z,dtype=float),self.lo_,self.hi_)
def make_preprocessor(cfg):
    scaler={'standard':StandardScaler,'minmax':MinMaxScaler,'robust':RobustScaler}[cfg['scaler']]()
    num=Pipeline([('imputer',SimpleImputer(strategy='median')),
                  ('clip',PercentileClipper(tuple(NUM),cfg['clipping'])),
                  ('scale',scaler)])
    try:onehot=OneHotEncoder(handle_unknown='ignore',sparse_output=False)
    except TypeError:onehot=OneHotEncoder(handle_unknown='ignore',sparse=False)
    cat=Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),
                  ('onehot',onehot)])
    return ColumnTransformer([('num',num,NUM),('cat',cat,CAT)],sparse_threshold=0)
def make_model(d,cfg):
    widths=[d]+list(cfg['hidden'])
    layers=[]
    for a,b in zip(widths[:-1],widths[1:]):
        layers.extend([nn.Linear(a,b),nn.ReLU() if cfg['activation']=='relu' else nn.Tanh()])
        if cfg['dropout']>0:layers.append(nn.Dropout(cfg['dropout']))
    layers.append(nn.Linear(widths[-1],1))
    return nn.Sequential(*layers)


## 3 — Üç sabit başlangıç seed'iyle final MLP modellerini eğit, model ve ön işleme dosyalarını kaydet


In [ ]:
def train_seed(seed):
    seed_everywhere(seed)
    started=time.perf_counter()
    fit_i,monitor_i=train_test_split(np.arange(len(Y_DEV)),test_size=0.15,
        stratify=Y_DEV,random_state=seed)
    prep=make_preprocessor(CFG)
    Zfit=np.asarray(prep.fit_transform(X_DEV.iloc[fit_i]),dtype=np.float32)
    Zmon=np.asarray(prep.transform(X_DEV.iloc[monitor_i]),dtype=np.float32)
    Ztest=np.asarray(prep.transform(X_TEST),dtype=np.float32)
    net=make_model(Zfit.shape[1],CFG).to(DEVICE)
    num_params=sum(p.numel() for p in net.parameters() if p.requires_grad)
    optimizer=CFG['optimizer']
    if optimizer=='sgd':
        opt=torch.optim.SGD(net.parameters(),lr=CFG['lr'],momentum=0.9,weight_decay=CFG['weight_decay'])
    elif optimizer=='adam':
        opt=torch.optim.Adam(net.parameters(),lr=CFG['lr'],weight_decay=CFG['weight_decay'])
    else:
        opt=torch.optim.AdamW(net.parameters(),lr=CFG['lr'],weight_decay=CFG['weight_decay'])
    scheduler=(torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode='min',
        factor=0.5,patience=5,min_lr=1e-5) if CFG['scheduler'] else None)
    criterion=nn.BCEWithLogitsLoss()
    loader=DataLoader(TensorDataset(torch.from_numpy(Zfit),
        torch.tensor(Y_DEV[fit_i],dtype=torch.float32)),
        batch_size=int(CFG['batch_size']),shuffle=True,
        generator=torch.Generator().manual_seed(int(seed)),num_workers=0)
    monitor_x=torch.from_numpy(Zmon).to(DEVICE)
    monitor_y=torch.tensor(Y_DEV[monitor_i],dtype=torch.float32,device=DEVICE)
    best=float('inf');best_epoch=0;best_state=None;bad=0;hist=[]
    for epoch in range(1,151):
        net.train();total=0.;count=0
        for xb,yb in loader:
            xb=xb.to(DEVICE);yb=yb.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            loss=criterion(net(xb).flatten(),yb)
            if not torch.isfinite(loss):raise RuntimeError('Geçersiz training loss')
            loss.backward();opt.step()
            total+=float(loss.item())*len(yb);count+=len(yb)
        net.eval()
        with torch.no_grad():mon_loss=float(criterion(net(monitor_x).flatten(),monitor_y).item())
        if not np.isfinite(mon_loss):raise RuntimeError('Geçersiz monitor loss')
        hist.append(dict(epoch=epoch,training_bce=total/count,
            monitor_bce=mon_loss,learning_rate=float(opt.param_groups[0]['lr'])))
        if mon_loss<best-1e-8:
            best=mon_loss;best_epoch=epoch;best_state=copy.deepcopy(net.state_dict());bad=0
        else:bad+=1
        if scheduler is not None:scheduler.step(mon_loss)
        if bad>=15:break
    assert best_state is not None
    net.load_state_dict(best_state);net.eval()
    probs=[]
    with torch.no_grad():
        for k in range(0,len(Ztest),1024):
            x=torch.from_numpy(Ztest[k:k+1024]).to(DEVICE)
            probs.extend(torch.sigmoid(net(x).flatten()).cpu().numpy().astype('float64').tolist())
    out=np.asarray(probs,dtype=np.float64)
    assert len(out)==len(TEST_IDS) and np.isfinite(out).all() and np.all((out>=0)&(out<=1))
    model_path=MODELS/f'mlp_seed{seed}.pt'
    prep_path=MODELS/f'preprocessor_seed{seed}.joblib'
    hist_path=HISTORIES/f'mlp_seed{seed}_history.csv'
    info_path=MODELS/f'mlp_seed{seed}_info.json'
    torch.save({'state_dict':{k:v.detach().cpu() for k,v in net.state_dict().items()},
        'input_features':int(Zfit.shape[1]),'model_config':CFG,
        'seed':seed,'best_epoch':best_epoch,'parameter_count':num_params},model_path)
    joblib.dump(prep,prep_path)
    write_csv(pd.DataFrame(hist),hist_path)
    info=dict(seed=seed,selected_id=FINAL['candidate'],best_epoch=best_epoch,
        epochs_run=len(hist),best_monitor_bce=best,monitor_metric='BCE',
        train_seconds=time.perf_counter()-started,parameter_count=num_params,
        fit_n=len(fit_i),monitor_n=len(monitor_i),code_version=CODE_VERSION,
        model_sha256=sha(model_path),preprocessor_sha256=sha(prep_path),
        history_sha256=sha(hist_path))
    save_json(info_path,info)
    return out,info
LOCK_CANDIDATE=RESULTS/'test_predictions_locked.csv'
LOCK_MANIFEST=RESULTS/'test_lock_manifest.json'
if LOCK_CANDIDATE.exists() or LOCK_MANIFEST.exists():
    raise RuntimeError('Kilit adayı zaten var. Aşama 3A tekrar çalıştırılmasın; mevcut dosyaları koruyun ve yükleyin.')
pred_frame=pd.DataFrame({'original_index':TEST_IDS.astype(int)})
model_records={}
for seed in SEED_LIST:
    probabilities,info=train_seed(seed)
    pred_frame[f'mlp_prob_seed{seed}']=probabilities
    pred_frame[f'mlp_pred_seed{seed}']=(probabilities>=float(FINAL['threshold'])).astype(int)
    model_records[str(seed)]=info
    print('MLP kaydedildi:',seed,'en iyi epoch:',info['best_epoch'])


## 4 — Referans modeller ve etiketsiz test tahmini dosyası


In [ ]:
baseline_cfg={'optimizer':'adamw','hidden':[32],'scaler':'standard','lr':0.001,
    'weight_decay':1e-4,'batch_size':64,'activation':'relu',
    'dropout':0.,'scheduler':True,'clipping':False}
reference_prep=make_preprocessor(baseline_cfg)
Zdev=reference_prep.fit_transform(X_DEV)
Ztest=reference_prep.transform(X_TEST)
references={'dummy':DummyClassifier(strategy='prior'),
            'logreg':LogisticRegression(max_iter=2000,random_state=31040)}
for name,model in references.items():
    with warnings.catch_warnings(record=True) as w:
        warnings.simplefilter('always')
        model.fit(Zdev,Y_DEV)
    probs=model.predict_proba(Ztest)[:,list(model.classes_).index(1)]
    pred_frame[f'{name}_prob']=probs
    pred_frame[f'{name}_pred']=(probs>=0.5).astype(int)
    joblib.dump(model,MODELS/f'{name}.joblib')
    print('Referans model:',name,'uyarı sayısı:',len(w))
joblib.dump(reference_prep,MODELS/'reference_preprocessor.joblib')
expected_cols=['original_index']
for seed in SEED_LIST:expected_cols.extend([f'mlp_prob_seed{seed}',f'mlp_pred_seed{seed}'])
expected_cols+=['dummy_prob','dummy_pred','logreg_prob','logreg_pred']
assert list(pred_frame.columns)==expected_cols
assert pred_frame['original_index'].is_unique and set(pred_frame['original_index'])==set(TEST_IDS)
assert not set(pred_frame.columns)&{'y','target','label','true_label','actual'}
assert pred_frame.notna().all().all()
write_csv(pred_frame,LOCK_CANDIDATE)
print('Etiketsiz tahmin dosyası:',LOCK_CANDIDATE)


## 5 — SHA-256 kilit manifesti ve dosya kontrolleri

Bu aşamada **GitHub commit kilidi henüz yoktur**. Gerçek test metriklerini hesaplamayın. Üç istenen dosyayı ChatGPT'ye yükleyin; GitHub üzerinden birebir doğrulamayı asistan yapacaktır.


In [ ]:
files={
  'results/test_predictions_locked.csv':sha(LOCK_CANDIDATE),
  'configs/final_hyperparameters.json':sha(CONFIGS/'final_hyperparameters.json'),
  'results/dataset_split.csv':sha(RESULTS/'dataset_split.csv'),
  'configs/stage2_plan.json':sha(CONFIGS/'stage2_plan.json'),
}
for p in sorted(MODELS.glob('*')):files[str(p.relative_to(ROOT))]=sha(p)
for p in sorted(HISTORIES.glob('*')):files[str(p.relative_to(ROOT))]=sha(p)
manifest=dict(project='YSA_31040_20261118',lock_status='PENDING_GITHUB_COMMIT',
    code_version=CODE_VERSION,base_seed=31040,primary_seed=SEED_LIST[0],
    model_seeds=SEED_LIST,threshold=float(FINAL['threshold']),
    selected_method=FINAL['method'],selected_candidate=FINAL['candidate'],
    stage2_plan_sha256=PLAN['plan_sha256'],raw_data_sha256=raw_sha,
    test_n=len(TEST_IDS),pred_columns=expected_cols,
    hash_algorithm='SHA-256 of raw file bytes; NOT Git blob or commit SHA',
    files=files,models=model_records,python=sys.version.split()[0],
    sklearn_version=sklearn.__version__,torch_version=torch.__version__,
    device=str(DEVICE))
save_json(LOCK_MANIFEST,manifest)
assert sha(LOCK_CANDIDATE)==manifest['files']['results/test_predictions_locked.csv']
assert sha(CONFIGS/'final_hyperparameters.json')==manifest['files']['configs/final_hyperparameters.json']
assert pd.read_csv(LOCK_CANDIDATE).shape==(len(TEST_IDS),len(expected_cols))
assert json.loads(LOCK_MANIFEST.read_text(encoding='utf-8'))['lock_status']=='PENDING_GITHUB_COMMIT'
print('AŞAMA 3A BAŞARILI — ETİKETSİZ TAHMİNLER HAZIR. GITHUB KİLİDİ HENÜZ YOK.')
print('LÜTFEN CHATGPT YE ŞU 3 DOSYAYI YÜKLEYİN:')
print('1.',LOCK_CANDIDATE)
print('2.',LOCK_MANIFEST)
print('3.',CONFIGS/'final_hyperparameters.json')
